# Baselines: Blind Recovery of Latent Domains

This notebook covers two tasks:

1. **Load and visualise pre-computed baseline results** stored in `baselines/*/results/`.
2. **Run a fresh baseline** (TICA or GLASSO) on waveform data and evaluate it.

The baselines implemented are:

| Folder | Method | Data |
|---|---|---|
| `tica-gsn` | TICA (Time-contrastive ICA) | GSN waveforms |
| `tica-ising` | TICA | Ising model samples |
| `tica-mnist` | TICA | MNIST crops |
| `glasso-waveform` | Graphical LASSO | GSN waveforms |
| `glasso-mnist` | Graphical LASSO | MNIST crops |
| `lgan-gsn` | Latent GAN | GSN waveforms |
| `manifold-mnist` | UMAP / Isomap | MNIST crops |
| `manifold-neuropixel` | UMAP / Isomap | Neuropixel spikes |

## 1. Setup

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

BASELINES_ROOT = os.path.join(REPO_ROOT, 'baselines')
print('Baselines directory:', BASELINES_ROOT)

## 2. Load and visualise pre-computed results

### 2a. GLASSO on 1-D waveforms

In [ ]:
glasso_results_dir = os.path.join(BASELINES_ROOT, 'glasso-waveform', 'results')
result_files = sorted(os.listdir(glasso_results_dir))
print('Available result files:')
for f in result_files:
    print(' ', f)

In [ ]:
def load_glasso_results(results_dir):
    """Load all GLASSO result JSONs from a results directory."""
    rows = []
    for fname in sorted(os.listdir(results_dir)):
        if not fname.endswith('.json'):
            continue
        with open(os.path.join(results_dir, fname)) as f:
            data = json.load(f)
        cfg   = data['config']
        summary = data['summary']
        best_key = data.get('best_configuration', None)
        for key, val in summary.items():
            rows.append({
                'file': fname,
                'feature': cfg.get('feature', '?'),
                'representation': cfg.get('output_representation', '?'),
                'baseline': val.get('baseline', key.split('_')[0]),
                'config_key': key,
                'is_best': key == best_key,
                'correlation_mean': val.get('correlation_mean_across_trials', np.nan),
                'frob_error_mean':  val.get('frob_error_mean_across_trials',  np.nan),
                'alpha': val.get('alpha', np.nan),
            })
    return rows


rows = load_glasso_results(glasso_results_dir)

# Group by (feature, representation) and plot correlation vs alpha
from itertools import groupby
from operator import itemgetter

groups = {}
for r in rows:
    if r['baseline'] != 'glasso':
        continue
    key = (r['feature'], r['representation'])
    groups.setdefault(key, []).append(r)

fig, axes = plt.subplots(1, len(groups), figsize=(6 * len(groups), 4), squeeze=False)
axes = axes[0]

for ax, ((feat, rep), group) in zip(axes, groups.items()):
    group_sorted = sorted(group, key=lambda r: r['alpha'] if not np.isnan(r['alpha']) else 0)
    alphas = [r['alpha'] for r in group_sorted]
    corrs  = [r['correlation_mean'] for r in group_sorted]
    best_corr = max((r['correlation_mean'] for r in group_sorted if r['is_best']), default=None)

    ax.semilogx(alphas, corrs, '-o', markersize=5)
    if best_corr is not None:
        ax.axhline(best_corr, ls='--', color='red', label=f'Best: {best_corr:.3f}')
        ax.legend(fontsize=9)
    ax.set_xlabel('GLASSO alpha (regularisation)')
    ax.set_ylabel('Mean Pearson correlation')
    ax.set_title(f'GLASSO | {feat} | {rep}')
    ax.grid(True, which='both', alpha=0.4)
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))

plt.suptitle('GLASSO: correlation vs regularisation strength (waveform data)')
plt.tight_layout(); plt.show()

### 2b. TICA experiments

In [ ]:
def summarise_tica_experiment(exp_json_path):
    """Print a summary table of a TICA experiments.json."""
    with open(exp_json_path) as f:
        experiments = json.load(f)
    print(f'Experiment file: {exp_json_path}')
    print(f'Number of configs: {len(experiments)}')
    for name, spec in list(experiments.items())[:3]:
        print(f'  {name}: pool_size={spec.get("pool_size")}, lr={spec.get("learning_rate")}, '
              f'epochs={spec.get("epochs")}, rep={spec.get("output_representation")}')
    print('  ...')

for subdir in ['tica-gsn', 'tica-ising', 'tica-mnist']:
    exp_path = os.path.join(BASELINES_ROOT, subdir, 'experiments.json')
    if os.path.exists(exp_path):
        summarise_tica_experiment(exp_path)
        print()

## 3. Run TICA on 1-D Gaussian waveforms

> **Note:** The paper reports that TICA consistently exhibited convergence failures across settings and is therefore not included in the main results tables. The cells below show how to run TICA for reference, but results should be interpreted with this caveat in mind.

The TICA baselines live in `baselines/tica-gsn/` and depend on a `synthetic_data_generator.py` local to that folder.  
We add that folder to the path temporarily to import it.

In [ ]:
import importlib, sys

TICA_GSN_DIR = os.path.join(BASELINES_ROOT, 'tica-gsn')
if TICA_GSN_DIR not in sys.path:
    sys.path.insert(0, TICA_GSN_DIR)

# Import from the baseline directory
import importlib.util

def import_from_path(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

tica_mod  = import_from_path('tica', os.path.join(TICA_GSN_DIR, 'tica.py'))
dgen_mod  = import_from_path('syn_dgen', os.path.join(TICA_GSN_DIR, 'synthetic_data_generator.py'))

TICAModel1D  = tica_mod.TICAModel1D
BaseDataGenerator = dgen_mod.DataGenerator
print('TICA and data generator imported.')

In [ ]:
import tensorflow as tf

# ── Hyperparameters ───────────────────────────────────────────────────────────
N_COMPONENTS   = 33
POOL_SIZE      = 3
LEARNING_RATE  = 1e-2
EPOCHS         = 100       # set higher (e.g. 1500) for paper-quality results
STEPS_PER_EPOCH = 200
BATCH_SIZE     = 250
SEED           = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Data ──────────────────────────────────────────────────────────────────────
features = [{"type": "gaussian", "scale_min": 0.5, "scale_max": 2.5,
             "amplitude_min": 0.5, "amplitude_max": 1.5}]

dg = BaseDataGenerator(
    batch_size=BATCH_SIZE,
    features=features,
    n_components=N_COMPONENTS,
    noise_normalized_std=0.05,
    seed=SEED
)

all_data = np.concatenate(
    [dg.sample_batch_of_data() for _ in range(STEPS_PER_EPOCH)], axis=0
).astype(np.float32)

print(f'Dataset shape: {all_data.shape}')

In [ ]:
# ── ZCA whitening ─────────────────────────────────────────────────────────────
def compute_zca(data, eps=1e-5):
    mean = data.mean(axis=0)
    centered = data - mean
    cov = (centered.T @ centered) / len(centered)
    vals, vecs = np.linalg.eigh(cov)
    W = vecs @ np.diag(1.0 / np.sqrt(vals + eps)) @ vecs.T
    return mean, W

mean_zca, W_zca = compute_zca(all_data)
data_whitened = (all_data - mean_zca) @ W_zca

# ── TICA model ────────────────────────────────────────────────────────────────
model_tica = TICAModel1D(
    input_dim=N_COMPONENTS,
    n_components=N_COMPONENTS,
    pool_size=POOL_SIZE,
    is_circulant=False
)

optimizer = tf.optimizers.Adam(LEARNING_RATE)
dataset   = tf.data.Dataset.from_tensor_slices(data_whitened).shuffle(50000).batch(BATCH_SIZE)

losses = []
for epoch in range(EPOCHS):
    epoch_loss = []
    for x_batch in dataset:
        with tf.GradientTape() as tape:
            loss = model_tica(x_batch, training=True)
        grads = tape.gradient(loss, model_tica.trainable_variables)
        optimizer.apply_gradients(zip(grads, model_tica.trainable_variables))
        epoch_loss.append(loss.numpy())
    losses.append(np.mean(epoch_loss))
    if epoch % 20 == 0:
        print(f'Epoch {epoch:3d} | loss = {losses[-1]:.4f}')

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('TICA loss')
plt.title('TICA training curve')
plt.grid(True); plt.tight_layout(); plt.show()

### Inspect learned TICA filters

In [ ]:
W_tica = model_tica.W.numpy()   # (input_dim, n_components)

fig, axes = plt.subplots(4, 8, figsize=(14, 6))
axes = axes.ravel()
for i in range(min(32, N_COMPONENTS)):
    axes[i].plot(W_tica[:, i])
    axes[i].set_title(f'Filter {i}', fontsize=7)
    axes[i].axis('off')

plt.suptitle('TICA learned filters (columns of W)')
plt.tight_layout(); plt.show()

## 4. Run GLASSO on 1-D Gaussian waveforms

In [ ]:
from sklearn.covariance import graphical_lasso

GLASSO_ALPHAS = [0.08, 0.16, 0.32]

# Use the same data (unwhitened)
cov_emp = np.cov(all_data.T)

glasso_results = {}
for alpha in GLASSO_ALPHAS:
    try:
        prec, cov_gl = graphical_lasso(cov_emp, alpha=alpha, max_iter=500)
        glasso_results[alpha] = {'precision': prec, 'covariance': cov_gl}
        print(f'alpha={alpha:.2f} | sparsity={np.mean(prec == 0):.2f}')
    except Exception as e:
        print(f'alpha={alpha:.2f} | FAILED: {e}')

In [ ]:
fig, axes = plt.subplots(1, len(glasso_results), figsize=(5 * len(glasso_results), 4))
if len(glasso_results) == 1:
    axes = [axes]

for ax, (alpha, res) in zip(axes, glasso_results.items()):
    im = ax.imshow(res['precision'], cmap='RdBu_r',
                   vmin=-np.abs(res['precision']).max(),
                   vmax= np.abs(res['precision']).max())
    ax.set_title(f'GLASSO precision matrix (alpha={alpha})')
    plt.colorbar(im, ax=ax)

plt.suptitle('GLASSO: learned precision matrices\n(near-tridiagonal = recovered 1-D Markov structure)')
plt.tight_layout(); plt.show()

## 5. Compare pre-computed GLASSO performance across conditions

In [ ]:
conditions = [
    ('gaussian', 'natural',  'gaussian_natural_glasso_1trials_results.json'),
    ('gaussian', 'dst',      'gaussian_dst_glasso_1trials_results.json'),
    ('legendre', 'natural',  'legendre_natural_glasso_1trials_results.json'),
    ('legendre', 'dst',      'legendre_dst_glasso_1trials_results.json'),
    ('ising',    'linear',   'ising_linear_glasso_1trials_results.json'),
]

summary_rows = []
for feat, rep, fname in conditions:
    fpath = os.path.join(BASELINES_ROOT, 'glasso-waveform', 'results', fname)
    if not os.path.exists(fpath):
        continue
    with open(fpath) as f:
        data = json.load(f)
    best_key = data.get('best_configuration')
    best = data['summary'].get(best_key, {})
    summary_rows.append({
        'condition': f'{feat}/{rep}',
        'best_alpha': best.get('alpha', '?'),
        'best_correlation': best.get('correlation_mean_across_trials', np.nan),
        'frob_error': best.get('frob_error_mean_across_trials', np.nan),
    })

print(f'{"Condition":<22}  {"Best alpha":>10}  {"Best corr":>10}  {"Frob err":>10}')
print('-' * 58)
for r in summary_rows:
    print(f'{r["condition"]:<22}  {str(r["best_alpha"]):>10}  {r["best_correlation"]:>10.4f}  {r["frob_error"]:>10.4f}')

# Bar chart
labels = [r['condition'] for r in summary_rows]
corrs  = [r['best_correlation'] for r in summary_rows]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, corrs, color='steelblue', alpha=0.8)
ax.set_ylabel('Best mean Pearson correlation')
ax.set_ylim(0, 1.0)
ax.set_title('GLASSO best performance across conditions')
ax.bar_label(bars, fmt='%.3f', fontsize=9)
plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()